# SLM Code Documentation — Evaluation Notebook
**Model:** Qwen2.5-Coder-1.5B-Instruct + LoRA adapters  
**Accelerator:** T4 x1 (or CPU)  
**Estimated Runtime:** 1–2 hours  

### Evaluation Metrics
1. **BLEU-4** — surface n-gram overlap
2. **ROUGE-L** — recall-oriented overlap
3. **BERTScore** — semantic similarity
4. **Param Coverage** — % of function parameters mentioned in generated doc
5. **Style Accuracy** — correct documentation style used
6. **Hallucination Rate** — references to non-existent identifiers
7. **Latency** — inference time p50/p95

### Comparison
- Fine-tuned model vs base model (no fine-tuning)
- Per-language breakdown
- Per-style breakdown

## Cell 1 — Install Dependencies

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install unsloth --quiet
!pip install rouge_score bert_score sacrebleu datasets --quiet
!pip install transformers peft --quiet

print("✅ Dependencies installed")

## Cell 2 — Imports and Configuration

In [ ]:
import unsloth
from unsloth import FastLanguageModel

import os, re, ast, json, time, warnings
import numpy as np
from pathlib import Path
from collections import defaultdict
from typing import Optional

import torch
from datasets import load_from_disk

from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
ADAPTER_DIR  = "/kaggle/working/slm_docgen_adapters"
DATASET_PATH = "/kaggle/input/datasets/srinidhichodavarapu/slm-docgen-dataset/slm_docgen_dataset/hf_dataset"
OUTPUT_DIR   = Path("/kaggle/working/slm_eval_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL       = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_NEW_TOKENS   = 256
EVAL_SAMPLES     = 300
SAMPLES_PER_LANG = EVAL_SAMPLES // 3
SEED             = 42

print(f"Eval samples     : {EVAL_SAMPLES} ({SAMPLES_PER_LANG} per language)")
print(f"Output dir       : {OUTPUT_DIR}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")

## Cell 3 — Load Evaluation Dataset

In [ ]:
import random
random.seed(SEED)

print("Loading validation split...")
dataset = load_from_disk(DATASET_PATH)
val_ds  = dataset["validation"]
print(f"Full validation set: {len(val_ds):,} samples")

# Stratified sample — equal per language
by_lang = defaultdict(list)
for i, sample in enumerate(val_ds):
    by_lang[sample["language"]].append(i)

eval_indices = []
for lang, indices in by_lang.items():
    sampled = random.sample(indices, min(SAMPLES_PER_LANG, len(indices)))
    eval_indices.extend(sampled)
    print(f"  {lang}: {len(sampled)} samples")

eval_subset = val_ds.select(eval_indices)
print(f"\nEval subset: {len(eval_subset)} samples")

## Cell 4 — Load Fine-tuned Model

In [ ]:
print("Loading fine-tuned model...")

ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_DIR,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
    cache_dir      = "/tmp/model_cache",
)
FastLanguageModel.for_inference(ft_model)
print("✅ Fine-tuned model loaded")
if torch.cuda.is_available():
    print(f"   VRAM used: {torch.cuda.memory_reserved()/1e9:.2f} GB")

## Cell 5 — Load Base Model (for comparison)

In [ ]:
print("Loading base model (no fine-tuning)...")

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
    cache_dir      = "/tmp/model_cache",
)
FastLanguageModel.for_inference(base_model)
print("✅ Base model loaded")

## Cell 6 — Inference Function

In [ ]:
SYSTEM_PROMPTS = {
    "python":     "You are a Python documentation assistant. Generate accurate, structured documentation following the specified style.",
    "java":       "You are a Java documentation assistant. Generate accurate Javadoc-style documentation.",
    "javascript": "You are a JavaScript documentation assistant. Generate accurate JSDoc-style documentation.",
}

def generate_doc(model, tokenizer, code: str, language: str, style: str):
    """Generate documentation and return (doc_string, latency_seconds)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS[language]},
        {"role": "user",   "content": (
            f"Language: {language}\n"
            f"Documentation style: {style}\n\n"
            f"```{language}\n{code}\n```\n\n"
            "Generate documentation for the above code."
        )},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start  = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = MAX_NEW_TOKENS,
            temperature        = 0.1,
            do_sample          = True,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )
    latency    = time.time() - start
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response, latency

print("✅ Inference function ready")

## Cell 7 — Metric Functions

In [ ]:
def compute_bleu(predictions, references):
    refs   = [[r] for r in references]
    result = sacrebleu.corpus_bleu(predictions, list(zip(*refs)))
    return round(result.score, 4)

def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = [scorer.score(ref, pred)["rougeL"].fmeasure
               for pred, ref in zip(predictions, references)]
    return round(np.mean(scores), 4)

def compute_bertscore(predictions, references):
    P, R, F = bert_score(
        predictions, references,
        model_type="distilbert-base-uncased",
        verbose=False,
    )
    return round(F.mean().item(), 4)

def extract_params(code, language):
    params = []
    if language == "python":
        try:
            tree = ast.parse(code)
            for node in ast.walk(tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    params = [a.arg for a in node.args.args
                              if a.arg not in ("self", "cls")]
                    break
        except:
            pass
    elif language in ("java", "javascript"):
        match = re.search(r"(?:function\s+\w+|\w+\s+\w+)\s*\(([^)]*)\)", code)
        if match:
            for param in match.group(1).split(","):
                name = re.split(r'[\s\[\]]+', param.strip())[-1]
                if name and not name.startswith("{"):
                    params.append(name)
    return params

def param_coverage(code, generated_doc, language):
    params = extract_params(code, language)
    if not params:
        return None
    mentioned = sum(1 for p in params if p in generated_doc)
    return round(mentioned / len(params), 4)

STYLE_MARKERS = {
    "google":           [r"Args:\s*\n", r"Returns:\s*\n"],
    "numpy":            [r"Parameters\s*\n\s*[-─]{3,}"],
    "restructuredtext": [r":param\s+\w+:", r":returns?:"],
    "javadoc":          [r"@param\s+\w+", r"@return"],
    "jsdoc":            [r"@param\s+\{?", r"@returns?"],
    "plain":            [],
}

def style_accuracy(generated_doc, expected_style):
    markers = STYLE_MARKERS.get(expected_style, [])
    if not markers:
        return True
    return any(re.search(m, generated_doc, re.IGNORECASE) for m in markers)

def hallucination_rate(code, generated_doc):
    code_tokens     = set(re.findall(r'\b[a-zA-Z_]\w{2,}\b', code))
    doc_identifiers = re.findall(
        r'\b[a-z][a-zA-Z_]*[A-Z_][a-zA-Z_]*\b|\b[a-z]+_[a-z_]+\b',
        generated_doc
    )
    if not doc_identifiers:
        return 0.0
    hallucinated = sum(1 for d in doc_identifiers if d not in code_tokens)
    return round(hallucinated / len(doc_identifiers), 4)

print("✅ Metric functions ready")

## Cell 8 — Run Evaluation on Fine-tuned Model

In [ ]:
print(f"Running evaluation on fine-tuned model ({len(eval_subset)} samples)...\n")

ft_results = []
for i, sample in enumerate(eval_subset):
    if i % 50 == 0:
        print(f"  {i}/{len(eval_subset)} samples done...")

    code       = sample["messages"][1]["content"]
    code_match = re.search(r"```\w*\n([\s\S]*?)```", code)
    code_clean = code_match.group(1).strip() if code_match else code
    reference  = sample["messages"][2]["content"]
    language   = sample["language"]
    style      = sample["style"]

    generated, latency = generate_doc(ft_model, ft_tokenizer, code_clean, language, style)

    ft_results.append({
        "language":       language,
        "style":          style,
        "code":           code_clean,
        "reference":      reference,
        "generated":      generated,
        "latency":        latency,
        "param_coverage": param_coverage(code_clean, generated, language),
        "style_accuracy": style_accuracy(generated, style),
        "hallucination":  hallucination_rate(code_clean, generated),
    })

print(f"\n✅ Fine-tuned model evaluation complete")

## Cell 9 — Run Evaluation on Base Model

In [ ]:
print(f"Running evaluation on base model ({len(eval_subset)} samples)...\n")

base_results = []
for i, sample in enumerate(eval_subset):
    if i % 50 == 0:
        print(f"  {i}/{len(eval_subset)} samples done...")

    code       = sample["messages"][1]["content"]
    code_match = re.search(r"```\w*\n([\s\S]*?)```", code)
    code_clean = code_match.group(1).strip() if code_match else code
    reference  = sample["messages"][2]["content"]
    language   = sample["language"]
    style      = sample["style"]

    generated, latency = generate_doc(base_model, base_tokenizer, code_clean, language, style)

    base_results.append({
        "language":       language,
        "style":          style,
        "code":           code_clean,
        "reference":      reference,
        "generated":      generated,
        "latency":        latency,
        "param_coverage": param_coverage(code_clean, generated, language),
        "style_accuracy": style_accuracy(generated, style),
        "hallucination":  hallucination_rate(code_clean, generated),
    })

print(f"\n✅ Base model evaluation complete")

## Cell 10 — Compute Corpus-Level Metrics

In [ ]:
def compute_all_metrics(results, label):
    predictions = [r["generated"] for r in results]
    references  = [r["reference"] for r in results]
    latencies   = [r["latency"]   for r in results]

    print(f"\nComputing metrics for {label}...")
    print("  BLEU...")
    bleu = compute_bleu(predictions, references)
    print("  ROUGE-L...")
    rouge = compute_rouge(predictions, references)
    print("  BERTScore (slowest)...")
    bertscore = compute_bertscore(predictions, references)

    pc_vals   = [r["param_coverage"] for r in results if r["param_coverage"] is not None]
    param_cov = round(np.mean(pc_vals), 4) if pc_vals else 0.0
    style_acc = round(np.mean([r["style_accuracy"] for r in results]), 4)
    hall_rate = round(np.mean([r["hallucination"]  for r in results]), 4)

    lat_sorted = sorted(latencies)
    p50 = round(lat_sorted[len(lat_sorted) // 2], 3)
    p95 = round(lat_sorted[int(len(lat_sorted) * 0.95)], 3)

    return {
        "label":          label,
        "bleu4":          bleu,
        "rouge_l":        rouge,
        "bertscore_f1":   bertscore,
        "param_coverage": param_cov,
        "style_accuracy": style_acc,
        "hallucination":  hall_rate,
        "latency_p50":    p50,
        "latency_p95":    p95,
        "n_samples":      len(results),
    }

ft_metrics   = compute_all_metrics(ft_results,   "Fine-tuned")
base_metrics = compute_all_metrics(base_results, "Base model")
print("\n✅ All metrics computed")

## Cell 11 — Results Summary

In [ ]:
def delta(ft_val, base_val, higher_better=True):
    diff  = ft_val - base_val
    arrow = ("↑" if diff > 0 else "↓") if higher_better else ("↓" if diff < 0 else "↑")
    return f"{arrow} {abs(diff):.4f}"

print("=" * 65)
print("  EVALUATION RESULTS")
print("=" * 65)
print(f"{'Metric':<22} {'Base':>10} {'Fine-tuned':>12} {'Delta':>12}")
print("-" * 65)

rows = [
    ("BLEU-4",           "bleu4",          True),
    ("ROUGE-L",          "rouge_l",         True),
    ("BERTScore F1",     "bertscore_f1",    True),
    ("Param Coverage",   "param_coverage",  True),
    ("Style Accuracy",   "style_accuracy",  True),
    ("Hallucination",    "hallucination",   False),
    ("Latency p50 (s)",  "latency_p50",     False),
    ("Latency p95 (s)",  "latency_p95",     False),
]
for name, key, hb in rows:
    bv = base_metrics[key]
    fv = ft_metrics[key]
    print(f"  {name:<20} {bv:>10.4f} {fv:>12.4f} {delta(fv, bv, hb):>12}")

print("=" * 65)
print(f"  Samples evaluated : {ft_metrics['n_samples']}")
print("=" * 65)

## Cell 12 — Per-Language Breakdown

In [ ]:
def per_language_metrics(results):
    by_lang = defaultdict(list)
    for r in results:
        by_lang[r["language"]].append(r)
    lang_metrics = {}
    for lang, lr in by_lang.items():
        preds = [r["generated"] for r in lr]
        refs  = [r["reference"] for r in lr]
        pc    = [r["param_coverage"] for r in lr if r["param_coverage"] is not None]
        lang_metrics[lang] = {
            "bleu4":          compute_bleu(preds, refs),
            "rouge_l":        compute_rouge(preds, refs),
            "param_coverage": round(np.mean(pc), 4) if pc else 0.0,
            "style_accuracy": round(np.mean([r["style_accuracy"] for r in lr]), 4),
            "n":              len(lr),
        }
    return lang_metrics

ft_lang   = per_language_metrics(ft_results)
base_lang = per_language_metrics(base_results)

print("=" * 70)
print("  PER-LANGUAGE BREAKDOWN")
print("=" * 70)
for lang in ["python", "java", "javascript"]:
    ft   = ft_lang.get(lang, {})
    base = base_lang.get(lang, {})
    print(f"\n  {lang.upper()} (n={ft.get('n', 0)})")
    print(f"  {'Metric':<20} {'Base':>10} {'Fine-tuned':>12}")
    print(f"  {'-'*45}")
    for metric in ["bleu4", "rouge_l", "param_coverage", "style_accuracy"]:
        bv = base.get(metric, 0)
        fv = ft.get(metric, 0)
        indicator = "✅" if fv >= bv else "⚠️"
        print(f"  {metric:<20} {bv:>10.4f} {fv:>12.4f}  {indicator}")
print("\n" + "=" * 70)

## Cell 13 — Qualitative Examples

In [ ]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
for r in ft_results:
    r["rouge_sample"] = scorer.score(r["reference"], r["generated"])["rougeL"].fmeasure

sorted_results = sorted(ft_results, key=lambda x: x["rouge_sample"], reverse=True)

print("=" * 70)
print("  TOP 2 EXAMPLES (highest ROUGE-L)")
print("=" * 70)
for r in sorted_results[:2]:
    print(f"\n  Language : {r['language']}  |  Style : {r['style']}")
    print(f"  ROUGE-L  : {r['rouge_sample']:.4f}  |  Param cov: {r['param_coverage']}")
    print(f"  {'─'*60}")
    print(f"  [REFERENCE]\n  {r['reference'][:300]}")
    print(f"\n  [GENERATED]\n  {r['generated'][:300]}")

print("\n" + "=" * 70)
print("  BOTTOM 2 EXAMPLES (lowest ROUGE-L)")
print("=" * 70)
for r in sorted_results[-2:]:
    print(f"\n  Language : {r['language']}  |  Style : {r['style']}")
    print(f"  ROUGE-L  : {r['rouge_sample']:.4f}  |  Param cov: {r['param_coverage']}")
    print(f"  {'─'*60}")
    print(f"  [REFERENCE]\n  {r['reference'][:300]}")
    print(f"\n  [GENERATED]\n  {r['generated'][:300]}")

## Cell 14 — Save Results

In [ ]:
full_report = {
    "overall":      {"fine_tuned": ft_metrics,   "base_model": base_metrics},
    "per_language": {"fine_tuned": ft_lang,       "base_model": base_lang},
    "config": {
        "eval_samples":    len(eval_subset),
        "samples_per_lang": SAMPLES_PER_LANG,
        "max_new_tokens":  MAX_NEW_TOKENS,
        "base_model":      BASE_MODEL,
        "adapter_dir":     ADAPTER_DIR,
    }
}

with open(OUTPUT_DIR / "eval_report.json", "w") as f:
    json.dump(full_report, f, indent=2)
print(f"✅ Report saved to {OUTPUT_DIR / 'eval_report.json'}")

with open(OUTPUT_DIR / "ft_sample_results.jsonl", "w") as f:
    for r in ft_results:
        f.write(json.dumps({
            "language":       r["language"],
            "style":          r["style"],
            "rouge_sample":   r.get("rouge_sample", 0),
            "param_coverage": r["param_coverage"],
            "style_accuracy": r["style_accuracy"],
            "hallucination":  r["hallucination"],
            "latency":        r["latency"],
        }) + "\n")
print(f"✅ Sample results saved")

print("\n📦 Output files:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print(f"   {f.name}  ({f.stat().st_size/1e3:.1f} KB)")